# Voxelization

Many CADD algorithms work with grid-like data structures,
e.g., to calculate energetic fields or to generate volumetric data such as binding pockets.
For this, the chemical system of interest (e.g., the receptor) must be voxelized (voxel: volumetric pixel),
i.e. cast from a point cloud into a grid.

This can be simply done by calling the `toxelate()` method of a `ChemicalSystem` object.

In [1]:
import sciapi
import scishow
import caddpy

## Example

In [2]:
PDB_ID = "3w32"

In [3]:
pdb_file = sciapi.pdb.file.entry(pdb_id=PDB_ID, file_format="pdb")

In [4]:
receptor = caddpy.chemsys.from_pdb(pdb_file)

Minimize axis-aligned bounding box to reduce the number of voxels:

In [5]:
receptor_minimized = receptor.minimize_aabb()

Voxelize:

In [6]:
voxel_field = receptor_minimized.toxelate(grid=0.3)

### Visualization

In [7]:
# Add the original receptor surface directly from its PDB file
nglwidget = scishow.nglview.NGLWidget().display(gui=True)
nglwidget.add_trajectory(receptor_minimized, name="Receptor")
nglwidget.clear_representations()   # Remove default representations
nglwidget.component_0.add_surface(  # Add surface representation
    color="rgb(100,20,20)",
    opacity=0.5,
    surface_type="vws",  # van der Waals surface
    scale_factor=0,      # do not scale
)

# Add the calculated voxel field
nglwidget.add_volume(
    voxel_field.tensor.astype(bool).astype(int),
    name="Voxel Field",
    basis=voxel_field.grid.unit_vectors,
    origin=voxel_field.grid.lower_bounds,
    representation_params=scishow.nglview.SurfaceRepresentationParameters(
        lazy=True,
        opacity=0.7,
        contour=False,
        color=(100,100,100),
        isolevel=0.5,  # Isolevel 0.5 for binary field shows the most accurate volume
        isolevel_type="value",
    )
)

ThemeManager()

NGLWidget(gui_style='ngl')